# Training on Kaggle: c-ResUNet-light (adopted) or c-ResUNet (Morelli et al., 2021)

One run trains one network on the 169 training images of `splits/green_split.json`; the checkpoint `best.pt` is the epoch
with the best global Dice on the 43 validation images.

| `MODEL` | network | extra option | output |
|---|---|---|---|
| `"light"` | c-ResUNet-light, adopted: no 5×5 residual block in the bottleneck (0.50 M parameters, receptive field ~96 px) | `--small_rf` | `runs/cresunet_light/` |
| `"cresunet"` | c-ResUNet of Morelli et al. (2021) (1.32 M parameters, receptive field ~160 px) | none | `runs/cresunet/` |

Recipe shared by both networks: 100 epochs without early stopping (`--patience 100`), 1200 random 512×512 crops per epoch,
batch 8, AdamW (learning rate 1e-3, weight decay 1e-4) with a cosine schedule, BCE + soft Dice loss, mixed precision, seed 42.
About 57 min on a T4. Output: `best.pt`, `last.pt`, `log.csv`, `config.json`.

Before running (the two input datasets are described in `kaggle/README.md`):
1. **Settings → Accelerator → GPU T4 x2**. Internet is not needed.
2. **Add Input**: the code dataset `cellseg-code` and the data dataset `fnc-green-data`.
3. Short check: `PROBE = True` (parameters cell) → **Run All**: 1 epoch on 8 training and 4 validation images (~1 min).
4. Full run: `PROBE = False` → **Save Version → Save & Run All (Commit)**; the commit keeps running with the browser closed.
5. Download the output folders from the **Output** tab of the version into the repository's `runs/`.

In [ ]:
# 1) GPU available?
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: Settings → Accelerator → 'GPU T4 x2'"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Find code and data under /kaggle/input, copy the code to /kaggle/working (/kaggle/input is read-only)
import os
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def find_one(pattern):
    '''First path under INPUT matching the pattern (Kaggle can nest dataset folders in different ways).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} (were both datasets added with 'Add Input'?)"
    return hits[0]

CODE_SRC = find_one("models/c_res_unet_small_rf.py").parent.parent
IMAGES = find_one("green/trainval/images")
MASKS = find_one("cleaned_masks/trainval/Green/masks")
TEST_IMAGES = find_one("green/test/images")
TEST_MASKS = find_one("cleaned_masks/test/Green/masks")
print("code:  ", CODE_SRC)
print("images:", IMAGES, "→", len(list(IMAGES.glob("*.png"))), "png (expected 212)")
print("masks: ", MASKS, "→", len(list(MASKS.glob("*.png"))), "png (expected 213: 128.png has no image)")

CODE = WORK / "code"
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)
os.chdir(CODE)
print("working dir:", os.getcwd())

## 3) Parameters

- `MODEL`: `"light"` (adopted network) or `"cresunet"` (Morelli et al., 2021).
- `PROBE = True`: short check; `PROBE = False`: full training.

In [ ]:
MODEL = "light"     # "light" (adopted) or "cresunet"
PROBE = True

MODELS = {"light": ("cresunet_light", ["--small_rf"]), "cresunet": ("cresunet", [])}
RUN_NAME, EXTRA = MODELS[MODEL]
RECIPE = ["--epochs", "100", "--patience", "100"]
LIMITS = []
if PROBE:
    RUN_NAME = "probe_" + RUN_NAME
    LIMITS = ["--epochs", "1", "--patience", "1", "--train_limit", "8", "--val_limit", "4"]
OUT = WORK / "runs" / RUN_NAME
print(MODEL, "→", OUT)

## 4) Training

In [ ]:
import subprocess
import sys
import time

def run(cmd):
    '''Runs a command, streams its output, stops on error.'''
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, f"failed (exit code {p.returncode})"

t0 = time.time()
run([sys.executable, "-m", "train.training", "--images_dir", IMAGES, "--masks_dir", MASKS,
     "--split_json", "splits/green_split.json", "--out_dir", OUT, "--num_workers", 4, *EXTRA, *RECIPE, *LIMITS])
print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 5) Learning curves and selected checkpoint
import json
import matplotlib.pyplot as plt
import pandas as pd

log = pd.read_csv(OUT / "log.csv")
cfg = json.load(open(OUT / "config.json"))
assert cfg["small_rf"] == (MODEL == "light")

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(log.epoch, log.train_loss, label="train")
ax[0].plot(log.epoch, log.val_loss, label="validation")
ax[0].set_title("loss (train on augmented crops, validation on whole images)")
ax[0].legend()
ax[1].plot(log.epoch, log.val_dice_global, label="global Dice")
ax[1].plot(log.epoch, log.val_dice_mean, label="mean Dice per image")
ax[1].set_title("validation Dice")
ax[1].legend()
ax[2].plot(log.epoch, log.lr)
ax[2].set_title("learning rate")
plt.tight_layout()
plt.show()

best = log.loc[log.val_dice_global.idxmax()]
print(f"best epoch {int(best.epoch)}: global Dice {best.val_dice_global:.4f} | global IoU {best.val_iou_global:.4f} | "
      f"mean Dice {best.val_dice_mean:.4f}")
print(f"mean time per epoch {log.time_s.mean():.0f} s | epochs run {len(log)}")